In [1]:
#Importing Libraries
import pandas as pd
import numpy as np
from scipy.stats import randint
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [3]:
# 1. Load Data & Clean
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)

In [4]:
# 2. Train / Test Split (80/20 Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [5]:
# 3. Baseline Model (Decision Tree)
dt_baseline = Pipeline([
    ('prep', preprocessor),
    ('clf', DecisionTreeClassifier(random_state=42))
])
dt_cv = cross_val_score(dt_baseline, X_train, y_train, cv=cv, scoring='accuracy').mean()
dt_baseline.fit(X_train, y_train)
dt_test = accuracy_score(y_test, dt_baseline.predict(X_test))

In [6]:
# 4. Hyperparameter Tuning: GridSearchCV
param_grid = {
    'clf__max_depth': [3, 5, 7, 10, 15, None],
    'clf__min_samples_split': [2, 5, 10, 20],
    'clf__min_samples_leaf': [1, 2, 4, 8],
    'clf__criterion': ['gini', 'entropy']
}
grid_search = GridSearchCV(
    estimator=Pipeline([('prep', preprocessor), ('clf', DecisionTreeClassifier(random_state=42))]),
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
grid_cv = grid_search.best_score_
grid_test = accuracy_score(y_test, grid_search.predict(X_test))

In [7]:
# 5. Hyperparameter Tuning: RandomizedSearchCV
param_dist = {
    'clf__max_depth': [3, 4, 5, 6, 7, 8, 10, 12, 15, None],
    'clf__min_samples_split': randint(2, 25),
    'clf__min_samples_leaf': randint(1, 15),
    'clf__criterion': ['gini', 'entropy']
}
rand_search = RandomizedSearchCV(
    estimator=Pipeline([('prep', preprocessor), ('clf', DecisionTreeClassifier(random_state=42))]),
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)
rand_search.fit(X_train, y_train)
rand_cv = rand_search.best_score_
rand_test = accuracy_score(y_test, rand_search.predict(X_test))

In [8]:
# 6. Ensemble Model (Random Forest)
rf_model = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_split=10, random_state=42, n_jobs=-1))
])
rf_cv = cross_val_score(rf_model, X_train, y_train, cv=cv, scoring='accuracy').mean()
rf_model.fit(X_train, y_train)
rf_test = accuracy_score(y_test, rf_model.predict(X_test))

In [11]:
# 7. Aggregate Comparison Table
summary_df = pd.DataFrame([
    {"Model": "Baseline (Decision Tree)", "CV Accuracy": f"{dt_cv:.4f}", "Test Accuracy": f"{dt_test:.4f}"},
    {"Model": "Decision Tree (GridSearchCV)", "CV Accuracy": f"{grid_cv:.4f}", "Test Accuracy": f"{grid_test:.4f}"},
    {"Model": "Decision Tree (RandomizedSearchCV)", "CV Accuracy": f"{rand_cv:.4f}", "Test Accuracy": f"{rand_test:.4f}"},
    {"Model": "Ensemble (Random Forest)", "CV Accuracy": f"{rf_cv:.4f}", "Test Accuracy": f"{rf_test:.4f}"
  }
])
print(summary_df.to_markdown(index=False))

| Model                              |   CV Accuracy |   Test Accuracy |
|:-----------------------------------|--------------:|----------------:|
| Baseline (Decision Tree)           |        0.7304 |          0.741  |
| Decision Tree (GridSearchCV)       |        0.7907 |          0.7942 |
| Decision Tree (RandomizedSearchCV) |        0.7907 |          0.7991 |
| Ensemble (Random Forest)           |        0.8019 |          0.8027 |


###Baseline Performance: The untuned Decision Tree establishes a baseline with 73.04% CV and 74.10% Test accuracy, showing the lowest predictive power across all tested setups.

###Hyperparameter Tuning Gains: Both tuning methods yielded an identical 79.07% CV accuracy (a +6.03% improvement over baseline CV). On the test set, RandomizedSearchCV slightly edged out GridSearchCV (79.91% vs. 79.42%), reflecting an overall test gain of +5.32% to +5.81% due to effective tree depth and sample split regularization.

###Ensemble Superiority: The Random Forest ensemble delivered the highest scores across both metrics (80.19% CV and 80.27% Test), achieving a +6.17% overall test accuracy improvement over the baseline while exhibiting the tightest gap between cross-validation and test performance (0.08%).